# Custom Layers

One factor behind deep learning's success
is the availability of a wide range of layers
that can be composed in creative ways
to design architectures suitable
for a wide variety of tasks.
For instance, researchers have invented layers
specifically for handling images, text,
looping over sequential data,
and
performing dynamic programming.
Sooner or later, you will need
a layer that does not exist yet in the deep learning framework.
In these cases, you must build a custom layer.
In this section, we show you how.


In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## (**Layers without Parameters**)

To start, we construct a custom layer
that does not have any parameters of its own.
This should look familiar if you recall our
introduction to modules in :numref:`sec_model_construction`.
The following `CenteredLayer` class simply
subtracts the mean from its input.
To build it, we simply need to inherit
from the base layer class and implement the forward propagation function.


In [2]:
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

Let's verify that our layer works as intended by feeding some data through it.


In [3]:
layer = CenteredLayer()
layer(torch.tensor([1.0, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

We can now [**incorporate our layer as a component
in constructing more complex models.**]


In [4]:
net = nn.Sequential(nn.LazyLinear(128), CenteredLayer())

As an extra sanity check, we can send random data
through the network and check that the mean is in fact 0.
Because we are dealing with floating point numbers,
we may still see a very small nonzero number
due to quantization.


In [5]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(-8.3819e-09, grad_fn=<MeanBackward0>)

## [**Layers with Parameters**]

Now that we know how to define simple layers,
let's move on to defining layers with parameters
that can be adjusted through training.
We can use built-in functions to create parameters, which
provide some basic housekeeping functionality.
In particular, they govern access, initialization,
sharing, saving, and loading model parameters.
This way, among other benefits, we will not need to write
custom serialization routines for every custom layer.

Now let's implement our own version of the  fully connected layer.
Recall that this layer requires two parameters,
one to represent the weight and the other for the bias.
In this implementation, we bake in the ReLU activation as a default.
This layer requires two input arguments: `in_units` and `units`, which
denote the number of inputs and outputs, respectively.


In [6]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

Next, we instantiate the `MyLinear` class
and access its model parameters.


In [7]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[-0.5519, -0.3220,  1.2418],
        [ 0.3559,  2.2357, -0.5870],
        [ 1.1292, -0.0048, -0.7610],
        [-1.5225, -0.2633,  0.1360],
        [-1.8041, -0.6308, -0.0188]], requires_grad=True)

We can [**directly carry out forward propagation calculations using custom layers.**]


In [8]:
linear(torch.rand(2, 5))

tensor([[0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.1040]])

We can also (**construct models using custom layers.**)
Once we have that we can use it just like the built-in fully connected layer.


In [9]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.],
        [0.]])

## Summary

We can design custom layers via the basic layer class. This allows us to define flexible new layers that behave differently from any existing layers in the library.
Once defined, custom layers can be invoked in arbitrary contexts and architectures.
Layers can have local parameters, which can be created through built-in functions.


## Exercises

1. Design a layer that takes an input and computes a tensor reduction,
   i.e., it returns $y_k = \sum_{i, j} W_{ijk} x_i x_j$.
1. Design a layer that returns the leading half of the Fourier coefficients of the data.


[Discussions](https://discuss.d2l.ai/t/59)



1. Design a layer that takes an input and computes a tensor reduction,
   i.e., it returns $y_k = \sum_{i, j} W_{ijk} x_i x_j$.


In [2]:
class TensorReductionLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # Create a weight tensor of shape (in_features, in_features, out_features)
        # This represents W_{ijk} where i,j are input indices and k is output index
        self.weight = nn.Parameter(torch.randn(in_features, in_features, out_features))
        
    def forward(self, x):
        # x shape: (batch_size, in_features)
        batch_size = x.shape[0]
        
        # Expand dimensions for broadcasting
        # x_i: (batch_size, in_features, 1)
        x_i = x.unsqueeze(2)
        # x_j: (batch_size, 1, in_features)
        x_j = x.unsqueeze(1)
        
        # Compute outer product of x with itself: x_i * x_j
        # Result shape: (batch_size, in_features, in_features)
        outer_product = x_i * x_j
        
        # Compute the reduction with the weights
        # Multiply the outer product with weights and sum over i,j dimensions
        # This computes sum_{i,j} W_{ijk} * x_i * x_j for each k
        y = torch.einsum('bij,ijk->bk', outer_product, self.weight)
        
        return y

NameError: name 'nn' is not defined

2. Design a layer that returns the leading half of the Fourier coefficients of the data.


# Designing a Fourier Transform Layer

This is an interesting challenge that combines signal processing with neural network design. Let me create a custom PyTorch layer that computes the Fourier transform of input data and returns only the leading half of the coefficients.

The Fourier transform converts data from the time/space domain to the frequency domain, and the "leading half" typically refers to the first half of the frequency components, which for real-valued inputs contains all the unique information (the second half is redundant due to symmetry properties).

Here's a complete implementation:

```python
class LeadingFourierLayer(nn.Module):
    def __init__(self):
        """
        A layer that returns the leading half of the Fourier coefficients.
        For real-valued inputs, this contains all the unique information.
        """
        super().__init__()
    
    def forward(self, x):
        """
        Parameters:
        -----------
        x : Tensor
            Input tensor with shape (batch_size, sequence_length)
            
        Returns:
        --------
        Tensor
            Leading half of the Fourier coefficients with shape 
            (batch_size, sequence_length // 2 + 1, 2)
            where the last dimension separates real and imaginary parts
        """
        # Apply Fast Fourier Transform
        fft_result = torch.fft.rfft(x, dim=1)
        
        # Convert complex numbers to real representation (real and imaginary parts)
        # Shape: (batch_size, sequence_length // 2 + 1, 2)
        real_repr = torch.stack([fft_result.real, fft_result.imag], dim=-1)
        
        return real_repr
```

## Testing the Layer

Let's test this layer with a simple example:

```python
# Create our Fourier layer
fourier_layer = LeadingFourierLayer()

# Create sample data with a known pattern (e.g., a sine wave)
batch_size = 2
seq_length = 16
t = torch.linspace(0, 2*torch.pi, seq_length)
x1 = torch.sin(t) + 0.5 * torch.sin(3*t)  # Fundamental + 3rd harmonic
x2 = torch.sin(2*t) + 0.3 * torch.sin(5*t)  # 2nd + 5th harmonic
x = torch.stack([x1, x2])

# Apply our Fourier layer
output = fourier_layer(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output (first sample):\n{output[0]}")
```

## Explanation and Intuition

The Fourier transform is a powerful mathematical tool that decomposes a signal into its constituent frequencies. It's like breaking down a musical chord into its individual notes:

1. A time-domain signal shows how a value changes over time
2. The Fourier transform shows which frequency components (like musical notes) make up that signal

In our implementation:

- We use PyTorch's `torch.fft.rfft()` function, which computes the "real FFT" - this is efficient for real-valued inputs because it only returns the non-redundant coefficients (about half the total).
- For a sequence of length N, the `rfft` returns N//2 + 1 complex coefficients.
- We separate the complex coefficients into real and imaginary parts for easier handling in neural networks.

### Why return only half the coefficients?

For real-valued inputs (which is common in most deep learning applications), the Fourier transform has symmetry - the second half of the coefficients are just complex conjugates of the first half. This means the second half contains no additional information, so we can discard it without losing anything.

This layer could be useful in:
- Audio processing, to focus on the most important frequency components
- Signal filtering, to remove noise or extract features
- Dimensionality reduction, by transforming data to the frequency domain and keeping only the most significant components

You could also extend this layer to make the number of retained coefficients a parameter, allowing you to control how much frequency information is preserved.